# 🩺 Diabetes Data Analysis
**Dataset:** Pima Indians Diabetes Database (Kaggle)

**Algorithms Used:**
- Logistic Regression
- Decision Tree Classifier

**Features:** Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age

**Target:** Outcome (0 = No Diabetes, 1 = Diabetes)

In [ ]:
# Install required libraries (if needed)
# !pip install pandas numpy scikit-learn matplotlib seaborn joblib

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
import joblib
import os

print('✅ All libraries imported successfully!')

## 2. Load Dataset
> Download from: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
> Place `diabetes.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv('diabetes.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())

# Target distribution
print('\nTarget Distribution:')
print(df['Outcome'].value_counts())

In [ ]:
# Target class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Outcome'].value_counts().plot(kind='bar', ax=axes[0], color=['#4CAF50','#F44336'])
axes[0].set_title('Outcome Distribution')
axes[0].set_xticklabels(['No Diabetes (0)', 'Diabetes (1)'], rotation=0)
axes[0].set_ylabel('Count')

df['Outcome'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                   colors=['#4CAF50','#F44336'], labels=['No Diabetes','Diabetes'])
axes[1].set_title('Outcome Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('static/outcome_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved!')

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdYlGn', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('static/correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Heatmap saved!')

In [ ]:
# Feature distributions
features = df.columns[:-1]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    df[df['Outcome']==0][col].plot(kind='hist', ax=axes[i], alpha=0.6, color='#4CAF50', label='No Diabetes')
    df[df['Outcome']==1][col].plot(kind='hist', ax=axes[i], alpha=0.6, color='#F44336', label='Diabetes')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('static/feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Distribution plots saved!')

## 4. Data Preprocessing

In [ ]:
# Replace zero values with median for columns where 0 is not valid
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in zero_invalid_cols:
    median_val = df[col].median()
    df[col] = df[col].replace(0, median_val)
    print(f'{col}: 0s replaced with median ({median_val:.2f})')

print('\n✅ Data cleaning complete!')

In [ ]:
# Split features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')
print('✅ Preprocessing done!')

## 5. Model Training

In [ ]:
# ── Logistic Regression ──
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_acc = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_proba)

print('=== Logistic Regression ===')
print(f'Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print(f'ROC-AUC:  {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['No Diabetes','Diabetes']))

In [ ]:
# ── Decision Tree ──
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)
dt_proba = dt_model.predict_proba(X_test_scaled)[:, 1]

dt_acc = accuracy_score(y_test, dt_pred)
dt_auc = roc_auc_score(y_test, dt_proba)

print('=== Decision Tree ===')
print(f'Accuracy: {dt_acc:.4f} ({dt_acc*100:.2f}%)')
print(f'ROC-AUC:  {dt_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, dt_pred, target_names=['No Diabetes','Diabetes']))

## 6. Model Evaluation & Visualizations

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, pred) in zip(axes, [('Logistic Regression', lr_pred), ('Decision Tree', dt_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Diabetes','Diabetes'],
                yticklabels=['No Diabetes','Diabetes'])
    ax.set_title(f'{name}\nConfusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('static/confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()
print('Confusion matrices saved!')

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))

for name, proba in [('Logistic Regression', lr_proba), ('Decision Tree', dt_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0,1],[0,1],'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('static/roc_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('ROC curves saved!')

In [ ]:
# Accuracy Comparison
models = ['Logistic Regression', 'Decision Tree']
accuracies = [lr_acc * 100, dt_acc * 100]
aucs = [lr_auc, dt_auc]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(models, accuracies, color=['#2196F3','#FF9800'], edgecolor='black')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.2f}%', ha='center', fontweight='bold')

bars2 = axes[1].bar(models, aucs, color=['#2196F3','#FF9800'], edgecolor='black')
axes[1].set_title('ROC-AUC Score Comparison')
axes[1].set_ylabel('AUC Score')
axes[1].set_ylim(0, 1)
for bar, val in zip(bars2, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('static/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Comparison chart saved!')

## 7. Feature Importance (Decision Tree)

In [ ]:
feature_imp = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
feature_imp.plot(kind='barh', color='#4CAF50', edgecolor='black')
plt.title('Feature Importance - Decision Tree')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('static/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Feature importance plot saved!')

## 8. Save Models

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(lr_model, 'models/logistic_regression.pkl')
joblib.dump(dt_model, 'models/decision_tree.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print('✅ Models saved successfully!')
print('📁 models/logistic_regression.pkl')
print('📁 models/decision_tree.pkl')
print('📁 models/scaler.pkl')

print(f'\n📊 Final Results Summary:')
print(f'{"Model":<25} {"Accuracy":>10} {"AUC":>8}')
print('-' * 45)
print(f'{"Logistic Regression":<25} {lr_acc*100:>9.2f}% {lr_auc:>8.4f}')
print(f'{"Decision Tree":<25} {dt_acc*100:>9.2f}% {dt_auc:>8.4f}')